# Perturbation Explorer

Visualises how each kinematic perturbation parameter shifts the focal spot across different sun positions.

**Workflow**
1. **Config** — pick a heliostat, set per-parameter-group deltas, choose number of sun positions.
2. **Load scenario** — loads the single-heliostat scenario (deflectometry surface, latest ARTIST).
3. **Load sun positions** — reads PAINT test samples; forward passes run on all, display selects `N_SUN_POSITIONS` evenly spread by elevation.
4. **Forward passes** — clean + one isolated pass per non-zero parameter group + combined (if >1 active).
5. **Grid** — rows = clean reference / isolated groups / combined; columns = sun positions.  
   Text overlay on each perturbed image shows the centroid shift in mrad.
6. **Scatter** — centroid shift vs. sun elevation per group.

In [15]:
import json, sys, pathlib, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import h5py
warnings.filterwarnings("ignore")

_nb   = pathlib.Path(globals().get("__vsc_ipynb_file__", pathlib.Path().resolve() / "x")).parent  # notebook dir
_src  = _nb.parent                 # .../src
_base = _src.parent                # .../master-thesis
sys.path.insert(0, str(_src))

from artist.raytracing.heliostat_ray_tracer import HeliostatRayTracer
from artist.io.paint_calibration_parser import PaintCalibrationDataParser
from artist.scenario.scenario import Scenario
from artist.util import get_device, set_logger_config
from artist.geometry import bitmap_coordinates_to_target_coordinates
from artist.flux import get_center_of_mass
import paint.util.paint_mappings as paint_mappings

from utils.synth_data import _forward_pass, apply_perturbations, reset_perturbations, _equalize_mapping
from utils.evaluation import build_heliostat_data_mapping

set_logger_config()
import logging; logging.getLogger().setLevel(logging.WARNING)
print("Imports OK")

Imports OK


In [16]:
# ============================================================
#  CONFIG — edit this cell, then re-run all cells below
# ============================================================

# Select one heliostat (ordered by increasing distance from the target)
HELIOSTAT_ID = "AC36"   # options: AC36  AG33  AO34  AW36  BE35

# Kinematic perturbation deltas added to the clean (unperturbed) parameters.
# Set all values to 0.0 for the unperturbed reference.
PERTURBATIONS = {
    # 4 joint-rotation deviations (rad)
    # [0] first_joint_tilt_n  [1] first_joint_tilt_u
    # [2] second_joint_tilt_e [3] second_joint_tilt_n
    "rotation_rad":       [0.05, 0.0, 0.0, 0.0],

    # 2 actuator initial-angle deviations a_i (rad)
    "actuator_angle_rad": [0.0, 0.0],

    # 2 actuator stroke-length deviations b_i (m)
    "actuator_stroke_m":  [0.0, 0.0],

    # 2 actuator offset deviations c_i (m)
    "actuator_offset_m":  [0.0, 0.0],

    # 9 joint + concentrator translation deviations (m)
    # [0-2] first_joint E/N/U  [3-5] second_joint E/N/U  [6-8] concentrator E/N/U
    "translation_m":      [0.0] * 9,

    # 3 heliostat base-position offset in ENU (m)
    "base_position_m":    [0.0, 0.0, 0.0],
}

N_RAYS                 = 100   # rays per light source (higher = cleaner, slower)
SURFACE_POINTS_PER_FACET = 5   # per facet (5×5 = 25 pts); increase for more surface detail

In [ ]:
%%time
# ── Load single-heliostat scenario ───────────────────────────────────────

SCENARIO_PATH = _base / "scenarios" / "one_heliostat_scenarios" / HELIOSTAT_ID / "scenario.h5"

device = get_device()
print(f"Device   : {device}")
print(f"Scenario : {SCENARIO_PATH}")

with h5py.File(SCENARIO_PATH, "r") as f:
    scenario = Scenario.load_scenario_from_hdf5(
        scenario_file=f,
        device=device,
        number_of_surface_points_per_facet=torch.tensor(
            [SURFACE_POINTS_PER_FACET, SURFACE_POINTS_PER_FACET]
        ),
    )

heliostat_group = scenario.heliostat_field.heliostat_groups[0]
heliostat_names = list(heliostat_group.names)
heliostat_idx   = 0   # always 0 for a single-heliostat scenario

hel_pos       = heliostat_group.positions[heliostat_idx, :3]
target_center = scenario.solar_tower.target_areas[0].centers[:, :3].mean(dim=0)
hel_dist_m    = torch.norm(hel_pos - target_center).item()

print(f"Heliostat: {HELIOSTAT_ID}  (index {heliostat_idx})")
print(f"Distance : {hel_dist_m:.1f} m  (heliostat → target center)")

In [ ]:
%%time
# ── Load test sun positions ───────────────────────────────────────────────

BENCHMARK_NAME = "benchmark_split-balanced_train-100_validation-50_deflectometry"
PAINT_DIR      = _base / "datasets" / "paint"
BENCHMARK_CSV  = PAINT_DIR / "splits" / f"{BENCHMARK_NAME}.csv"
CAL_DIR        = PAINT_DIR / BENCHMARK_NAME / "calibration_properties"
FLUX_DIR       = PAINT_DIR / BENCHMARK_NAME / "flux_image"

full_mapping = build_heliostat_data_mapping(
    benchmark_csv=BENCHMARK_CSV,
    calibration_properties_dir=CAL_DIR,
    flux_image_dir=FLUX_DIR,
    split="test",
)

heli_mapping = [(hid, cal, flux) for hid, cal, flux in full_mapping if hid == HELIOSTAT_ID]
if not heli_mapping:
    raise RuntimeError(f"{HELIOSTAT_ID} not found in test split")

n_samples = len(heli_mapping[0][1])
equalized  = _equalize_mapping(heli_mapping, sample_limit=n_samples)

parser = PaintCalibrationDataParser(
    sample_limit=n_samples,
    centroid_extraction_method=paint_mappings.UTIS_KEY,
)
with torch.no_grad():
    _, _, incident_rays, _, active_mask, target_mask = parser.parse_data_for_reconstruction(
        heliostat_data_mapping=equalized,
        heliostat_group=heliostat_group,
        scenario=scenario,
        device=device,
    )

# Select N_SUN_POSITIONS display indices evenly spread by elevation
sun_el_all  = torch.asin(-incident_rays[:, 2].clamp(-1, 1)).rad2deg()
sorted_idx  = torch.argsort(sun_el_all)
pick        = torch.linspace(0, len(sorted_idx) - 1, N_SUN_POSITIONS).long()
display_idx = sorted_idx[pick]   # indices into incident_rays for column display

print(f"Test samples loaded : {n_samples}")
print(f"Display sun elevations: {sun_el_all[display_idx].cpu().numpy().round(1)} °")


In [ ]:
%%time
# ── Forward passes: clean + one isolated pass per active group + combined ─

def _is_nonzero(v) -> bool:
    return any(x != 0.0 for x in v)

def _build_pert_tensors(group_name: str | None, n_heli: int, heli_idx: int, device) -> dict:
    """Return perturbation tensors with only `group_name` set (None → all groups)."""
    key_map = {
        "rotation_rad":       "rotation",
        "actuator_angle_rad": "actuator_angle",
        "actuator_stroke_m":  "actuator_stroke",
        "actuator_offset_m":  "actuator_offset",
        "translation_m":      "translation",
        "base_position_m":    "base_position",
    }
    pert = {
        "rotation":        torch.zeros(n_heli, 4, device=device),
        "actuator_angle":  torch.zeros(n_heli, 2, device=device),
        "actuator_stroke": torch.zeros(n_heli, 2, device=device),
        "actuator_offset": torch.zeros(n_heli, 2, device=device),
        "translation":     torch.zeros(n_heli, 9, device=device),
        "base_position":   torch.zeros(n_heli, 3, device=device),
    }
    for cfg_key, tensor_key in key_map.items():
        if group_name is None or cfg_key == group_name:
            pert[tensor_key][heli_idx] = torch.tensor(
                PERTURBATION_DELTAS[cfg_key], dtype=torch.float32, device=device
            )
    return pert


scenario.set_number_of_rays(N_RAYS)
n_heli    = len(heliostat_names)
zero_base = torch.zeros(n_heli, 3, device=device)

# Identify which groups have non-zero deltas
active_groups = [k for k, v in PERTURBATION_DELTAS.items() if _is_nonzero(v)]
print(f"Active perturbation groups: {active_groups or ['(none)']}")

# ── Clean pass ────────────────────────────────────────────────────────────
with torch.no_grad():
    clean_centroids, clean_flux = _forward_pass(
        scenario, heliostat_group, incident_rays, active_mask, target_mask,
        zero_base, device,
    )

# ── Per-group isolated passes ─────────────────────────────────────────────
group_results = {}   # group_name → {"centroids": ..., "flux": ..., "shifts_mrad": ...}
for gname in active_groups:
    pert_t   = _build_pert_tensors(gname, n_heli, heliostat_idx, device)
    snapshot = apply_perturbations(heliostat_group.kinematics, pert_t, device)
    with torch.no_grad():
        gc_c, gf = _forward_pass(
            scenario, heliostat_group, incident_rays, active_mask, target_mask,
            pert_t["base_position"], device,
        )
    reset_perturbations(heliostat_group.kinematics, snapshot)
    shifts_m    = torch.norm(gc_c[:, :3] - clean_centroids[:, :3], dim=1)
    shifts_mrad = shifts_m / hel_dist_m * 1000
    group_results[gname] = {"centroids": gc_c, "flux": gf, "shifts_mrad": shifts_mrad}
    print(f"  {gname:22s}: mean={shifts_mrad.mean():.1f} mrad  max={shifts_mrad.max():.1f} mrad")

# ── Combined pass (if more than one active group) ─────────────────────────
combined_result = None
if len(active_groups) > 1:
    pert_t   = _build_pert_tensors(None, n_heli, heliostat_idx, device)
    snapshot = apply_perturbations(heliostat_group.kinematics, pert_t, device)
    with torch.no_grad():
        cc_c, cf = _forward_pass(
            scenario, heliostat_group, incident_rays, active_mask, target_mask,
            pert_t["base_position"], device,
        )
    reset_perturbations(heliostat_group.kinematics, snapshot)
    shifts_m    = torch.norm(cc_c[:, :3] - clean_centroids[:, :3], dim=1)
    shifts_mrad = shifts_m / hel_dist_m * 1000
    combined_result = {"centroids": cc_c, "flux": cf, "shifts_mrad": shifts_mrad}
    print(f"  {'combined':22s}: mean={shifts_mrad.mean():.1f} mrad  max={shifts_mrad.max():.1f} mrad")

print("Forward passes complete.")


In [ ]:
# ── Grid: rows = clean / per-group / combined, columns = sun positions ────

def _to_norm(flux: torch.Tensor) -> np.ndarray:
    f = flux.cpu().float()
    fmax = f.max()
    return (f / fmax).numpy() if fmax > 1e-12 else f.numpy()

def _param_label(gname: str) -> str:
    vals = PERTURBATION_DELTAS[gname]
    nonzero = [(i, v) for i, v in enumerate(vals) if v != 0.0]
    detail = ", ".join(f"[{i}]={v:+.3g}" for i, v in nonzero)
    return f"{gname}\n{detail}"


# Row list: (label, flux_tensor | None, shifts_mrad | None)
rows = [("clean\nreference", None, None)]
for gname in active_groups:
    rows.append((_param_label(gname), group_results[gname]["flux"], group_results[gname]["shifts_mrad"]))
if combined_result is not None:
    rows.append(("combined", combined_result["flux"], combined_result["shifts_mrad"]))

n_rows = len(rows)
n_cols = N_SUN_POSITIONS
sun_el_np = sun_el_all.cpu().numpy()

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(n_cols * 2.0, n_rows * 2.2),
    squeeze=False,
)
fig.suptitle(
    f"{HELIOSTAT_ID} — perturbation effect per sun position\n"
    f"N_RAYS={N_RAYS}  |  columns ordered by sun elevation (low → high)",
    fontsize=11, y=1.01,
)

for r, (label, pert_flux, shifts_mrad) in enumerate(rows):
    axes[r, 0].set_ylabel(label, fontsize=7, rotation=0, labelpad=4,
                          ha="right", va="center")

    for c, di in enumerate(display_idx):
        ax  = axes[r, c]
        di  = di.item()
        el  = sun_el_np[di]

        if pert_flux is None:
            # Clean reference row
            ax.imshow(_to_norm(clean_flux[di]), cmap="inferno", vmin=0, vmax=1)
            ax.set_title(f"el={el:.1f}°", fontsize=7)
        else:
            ax.imshow(_to_norm(pert_flux[di]), cmap="inferno", vmin=0, vmax=1)
            shift_val = shifts_mrad[di].item()
            ax.text(0.97, 0.03, f"{shift_val:.1f} mrad",
                    transform=ax.transAxes, fontsize=6, color="white",
                    ha="right", va="bottom",
                    bbox=dict(boxstyle="round,pad=0.1", fc="black", alpha=0.4))
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ── Scatter: centroid shift (mrad) vs. sun elevation per group ────────────

sun_el_np = sun_el_all.cpu().numpy()

fig, ax = plt.subplots(figsize=(11, 4))

colors = plt.cm.tab10.colors
for idx, gname in enumerate(active_groups):
    smrad = group_results[gname]["shifts_mrad"].cpu().numpy()
    ax.scatter(sun_el_np, smrad, label=gname, color=colors[idx % 10],
               s=40, alpha=0.8, zorder=3)
    ax.axhline(smrad.mean(), ls="--", color=colors[idx % 10], alpha=0.5, lw=1)

if combined_result is not None and len(active_groups) > 1:
    smrad = combined_result["shifts_mrad"].cpu().numpy()
    ax.scatter(sun_el_np, smrad, label="combined", color="black",
               s=40, marker="D", alpha=0.7, zorder=4)
    ax.axhline(smrad.mean(), ls=":", color="black", alpha=0.5, lw=1)

ax.set_xlabel("Sun elevation (°)")
ax.set_ylabel("Centroid shift (mrad)")
ax.set_title(f"{HELIOSTAT_ID} — focal-spot shift vs. sun elevation per perturbation group")
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
